In [ ]:
%config InlineBackend.figure_formats = ['svg']
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
import scipy.sparse.linalg as spla
from functools import reduce

In [ ]:
plt.style.use("dark_background")
# mpl.rcParams["figure.facecolor"] = 'none'
mpl.rcParams["axes.facecolor"] = "none"
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["figure.figsize"] = [12.8, 9.6]

# Notes

Spec for mixed BC:

$$\begin{align*}
    -\triangle u &= f,            \qquad x \in \Omega \\
    \alpha u + \partial_n u &= g, \qquad x \in \partial \Omega
\end{align*}$$

In [ ]:
from collections import namedtuple

Dimension = namedtuple(
    "Dimension", ["low", "high", "num_divs", "low_robin", "high_robin"]
)
BoxDomain = namedtuple("BoxDomain", ["dimensions"])

In [ ]:
def fdm_neg_lapl_1d_mat(dim: Dimension):
    interior_size = dim.num_divs - 1
    full_size = (
        interior_size + int(dim.low_robin is not None) + int(dim.high_robin is not None)
    )
    dim_len = dim.high - dim.low
    h = dim_len / dim.num_divs
    invhsqr = 1 / (h * h)

    return (
        sparse.block_array(
            (
                [
                    [
                        sparse.csr_array(
                            ([2.0 * (1 + h * dim.low_robin), -2.0], [0, 1], [0, 2]),
                            shape=(1, full_size),
                        )
                    ]
                ]
                if dim.low_robin is not None
                else [[None]]
            )
            + (
                [
                    [
                        sparse.diags_array(
                            [2.0, -1, -1],
                            offsets=np.array([0, 1, -1])
                            + int(dim.low_robin is not None),
                            shape=(interior_size, full_size),
                        )
                    ]
                ]
            )
            + (
                [
                    [
                        sparse.csr_array(
                            (
                                [-2.0, 2.0 * (1 + h * dim.high_robin)],
                                [full_size - 2, full_size - 1],
                                [0, 2],
                            ),
                            shape=(1, full_size),
                        )
                    ]
                ]
                if dim.high_robin is not None
                else [[None]]
            )
        )
        * invhsqr
    )


def fdm_neg_lapl_unif_disc_mat(dom: BoxDomain):
    return reduce(
        sparse.kronsum,
        (fdm_neg_lapl_1d_mat(dim) for dim in reversed(dom.dimensions)),
    )


fig, ax = plt.subplots()
ax: plt.Axes = ax

ax_im = ax.matshow(
    fdm_neg_lapl_unif_disc_mat(
        BoxDomain(
            [
                Dimension(0.0, 1.0, 3, None, None),
                Dimension(0.0, 1.0, 4, 0.0, None),
                Dimension(0.0, 1.0, 5, None, 0.0),
            ]
        )
    ).toarray(),
    norm=mpl.colors.CenteredNorm(),
    cmap="berlin",
)
fig.colorbar(ax_im)
plt.show()

In [ ]:
def make_coordinate_grids(dom: BoxDomain):
    return np.meshgrid(
        *(
            np.linspace(dim.low, dim.high, num=dim.num_divs + 1)[
                slice(
                    1 if dim.low_robin is None else 0,
                    -1 if dim.high_robin is None else None,
                )
            ]
            for dim in dom.dimensions
        ),
        indexing="ij"
    )

In [ ]:
def manufactured_boundary_terms(dom: BoxDomain, coords, soln, grad):
    bd_terms = np.zeros_like(coords[0])
    num_dims = len(dom.dimensions)
    for i, dim in enumerate(dom.dimensions):
        dim: Dimension = dim
        dim_len = dim.high - dim.low
        h = dim_len / dim.num_divs
        invh = 1.0 / h
        invhsqr = invh * invh

        for face_idx, robin, bd_coord, normal_sign in [
            (0, dim.low_robin, dim.low, -1.0),
            (-1, dim.high_robin, dim.high, 1.0),
        ]:
            idx_selector = tuple(
                face_idx if i == j else slice(None) for j in range(num_dims)
            )
            face_coords = tuple(
                bd_coord if i == j else grid[idx_selector]
                for j, grid in enumerate(coords)
            )
            if robin is None:
                # Dirichlet
                bd_terms[idx_selector] += invhsqr * soln(face_coords)
            else:
                # Robin
                bd_terms[idx_selector] += (
                    2.0
                    * invh
                    * (robin * soln(face_coords) + normal_sign * grad(face_coords)[i])
                )

    return bd_terms

In [ ]:
def exact_soln(coords):
    x, y = coords
    return np.sin(x + 1.0) * np.sin(y + 2.0)


def exact_grad(coords):
    x, y = coords
    dxg = np.cos(x + 1.0) * np.sin(y + 2.0)
    dyg = np.sin(x + 1.0) * np.cos(y + 2.0)
    return np.stack([dxg, dyg])


def forcing(coords):
    return 2.0 * exact_soln(coords)


domain = BoxDomain(
    [
        Dimension(0.0, 1.0, 4, None, 0.0),
        Dimension(0.0, 1.0, 4, 0.0, None),
    ]
)

coords = make_coordinate_grids(domain)
poisson_lhs = fdm_neg_lapl_unif_disc_mat(domain)
poisson_rhs = (
    forcing(coords)
    + manufactured_boundary_terms(domain, coords, exact_soln, exact_grad)
).ravel()

approx = spla.spsolve(poisson_lhs, poisson_rhs)
error = np.max(exact_soln(coords) - approx.reshape(coords[0].shape))
error